In [1]:
# Cell 1: Imports, Spark load, join geography, convert to Pandas
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from utils.spark_session import get_spark
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

spark = get_spark()

GOLD_FACT = '../data/gold/fact/hospital_financials'
GOLD_DIM_G = '../data/gold/dims/dim_geography'

# Load the fact table
df_spark = spark.read.format('delta').load(GOLD_FACT)

# Load dim_geography — we need rural_level and STATE_CODE for the fairness analysis
df_geo = spark.read.format('delta').load(GOLD_DIM_G)

# Left join: attach rural_level and STATE_CODE to the fact table via geo_id
df_spark = df_spark.join(
    df_geo.select('geo_id', 'rural_level', 'STATE_CODE'),
    on='geo_id',
    how='left'
)

# Convert to Pandas — XGBoost and Scikit-learn work on DataFrames, not Spark DFs
df = df_spark.toPandas()

print(f'Total rows: {len(df)}')
print(f'Fiscal years present: {sorted(df["fiscal_year"].unique())}')
print(f'Distress label distribution:\n{df["distress_label"].value_counts()}')
print(f'Distress rate: {df["distress_label"].mean():.4f}')
print(f'\nrural_level distribution:\n{df["rural_level"].value_counts()}')

26/04/08 11:45:18 WARN Utils: Your hostname, Arjuns-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.234 instead (on interface en0)
26/04/08 11:45:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/arjunpillai/.ivy2/cache
The jars for the packages stored in: /Users/arjunpillai/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1e530a5a-4991-44f4-9b70-1337f1b590c0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central


:: loading settings :: url = jar:file:/Users/arjunpillai/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 106ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-1e530a5a-4991-44f4-9b70-1337f1b590c0
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
26/04/08 11:45:18 WARN NativeCodeLoader: Unable to load native-hado

Total rows: 24634
Fiscal years present: [np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
Distress label distribution:
distress_label
0    24559
1       75
Name: count, dtype: int64
Distress rate: 0.0030

rural_level distribution:
rural_level
Small Town      13956
Micropolitan     6166
Remote Rural     4512
Name: count, dtype: int64


In [2]:
# Cell 2: Define features and temporal train/val/test split

FEATURE_COLS = [
    'operating_margin',
    'total_margin',
    'cost_to_revenue',
    'medicaid_pct',
    'medicare_day_pct',
    'occupancy_proxy',
    'days_cash_on_hand',
    'current_ratio',
    'uncompensated_care_pct',
    'revenue_yoy_change'
]

TARGET = 'distress_label'

# Exclude 2023-2024: real CMS data but outside the reference doc's
# intended evaluation window. Keeping them would contaminate the test set
# with years the model was never designed to evaluate against.
df_model = df[df['fiscal_year'] <= 2022].copy()

print(f'Rows after excluding 2023-2024: {len(df_model)}')
print(f'Positive-class rows remaining: {df_model[TARGET].sum()}')

# Temporal split — never random for time-series prediction
TRAIN_YEARS = list(range(2011, 2019))   # 2011-2018: 8 years of history
VAL_YEARS   = [2019, 2020]              # Tune hyperparameters here
TEST_YEARS  = [2021, 2022]              # Final evaluation — untouched until end

train = df_model[df_model['fiscal_year'].isin(TRAIN_YEARS)]
val   = df_model[df_model['fiscal_year'].isin(VAL_YEARS)]
test  = df_model[df_model['fiscal_year'].isin(TEST_YEARS)]

X_train, y_train = train[FEATURE_COLS].copy(), train[TARGET].copy()
X_val,   y_val   = val[FEATURE_COLS].copy(),   val[TARGET].copy()
X_test,  y_test  = test[FEATURE_COLS].copy(),  test[TARGET].copy()

# Keep geography columns for the fairness analysis in Cell 6
rural_test  = test['rural_level'].reset_index(drop=True)
state_test  = test['STATE_CODE'].reset_index(drop=True)

print(f'\nTrain: {len(X_train)} rows | Positives: {y_train.sum()} | Rate: {y_train.mean():.4f}')
print(f'Val:   {len(X_val)} rows | Positives: {y_val.sum()} | Rate: {y_val.mean():.4f}')
print(f'Test:  {len(X_test)} rows | Positives: {y_test.sum()} | Rate: {y_test.mean():.4f}')

Rows after excluding 2023-2024: 22037
Positive-class rows remaining: 73

Train: 14720 rows | Positives: 58 | Rate: 0.0039
Val:   3679 rows | Positives: 3 | Rate: 0.0008
Test:  3638 rows | Positives: 12 | Rate: 0.0033


In [3]:
# Cell 3: Winsorization and imputation of known data quality issues

# ── WINSORIZATION ──────────────────────────────────────────────────────────
# medicaid_pct: max ~90 due to CMS gross charge scaling artifact.
# current_ratio: mean ~1,238, max ~9.9M due to near-zero liability denominators.
# Winsorize both at the 99th percentile of the TRAINING SET ONLY.
# We compute thresholds on train, then apply the same thresholds to val and test.
# Applying val/test statistics would leak information from those sets into preprocessing.

for col in ['medicaid_pct', 'current_ratio']:
    cap = X_train[col].quantile(0.99)
    X_train[col] = X_train[col].clip(upper=cap)
    X_val[col]   = X_val[col].clip(upper=cap)
    X_test[col]  = X_test[col].clip(upper=cap)
    print(f'{col}: 99th percentile cap = {cap:.4f}')

# ── IMPUTATION ─────────────────────────────────────────────────────────────
# revenue_yoy_change: null for each hospital's first observed fiscal year (~2,212 nulls)
# current_ratio: ~1,000 nulls from zero-liability rows
# Impute with the TRAINING SET median. Same logic: compute on train, apply everywhere.

impute_cols = ['revenue_yoy_change', 'current_ratio']
impute_medians = {}

for col in impute_cols:
    median_val = X_train[col].median()
    impute_medians[col] = median_val
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col]   = X_val[col].fillna(median_val)
    X_test[col]  = X_test[col].fillna(median_val)
    print(f'{col}: imputed nulls with training median = {median_val:.4f}')

# Verify no nulls remain across any split
for name, X in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    total_nulls = X.isnull().sum().sum()
    print(f'{name} remaining nulls: {total_nulls}')

# ── IMPUTATION CONTINUED ───────────────────────────────────────────────────
# operating_margin, total_margin, cost_to_revenue: nulls from zero-revenue rows.
# All three share TOTAL_REVENUES as denominator — they fail together.

revenue_ratio_cols = ['operating_margin', 'total_margin', 'cost_to_revenue']

for col in revenue_ratio_cols:
    median_val = X_train[col].median()
    impute_medians[col] = median_val
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col]   = X_val[col].fillna(median_val)
    X_test[col]  = X_test[col].fillna(median_val)
    print(f'{col}: imputed nulls with training median = {median_val:.4f}')

# Re-verify
for name, X in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    total_nulls = X.isnull().sum().sum()
    print(f'{name} remaining nulls: {total_nulls}')

medicaid_pct: 99th percentile cap = 1.2001
current_ratio: 99th percentile cap = 21.8335
revenue_yoy_change: imputed nulls with training median = 0.0258
current_ratio: imputed nulls with training median = 2.2259
Train remaining nulls: 27
Val remaining nulls: 15
Test remaining nulls: 9
operating_margin: imputed nulls with training median = -0.0510
total_margin: imputed nulls with training median = 0.0214
cost_to_revenue: imputed nulls with training median = 0.8955
Train remaining nulls: 0
Val remaining nulls: 0
Test remaining nulls: 0


In [4]:
# Cell 4: SMOTE oversampling on training set
from imblearn.over_sampling import SMOTE

# SMOTE generates synthetic positive-class samples by interpolating between
# existing minority-class rows in feature space. Applied to training only —
# val and test must reflect the real class distribution to give honest metrics.
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE — Train rows: {len(X_train)} | Positives: {y_train.sum()}')
print(f'After SMOTE  — Train rows: {len(X_train_sm)} | Positives: {y_train_sm.sum()}')
print(f'Class distribution after SMOTE:\n{pd.Series(y_train_sm).value_counts()}')

Before SMOTE — Train rows: 14720 | Positives: 58
After SMOTE  — Train rows: 29324 | Positives: 14662
Class distribution after SMOTE:
distress_label
0    14662
1    14662
Name: count, dtype: int64


In [5]:
# Cell 5: Modeling imports
import mlflow
import mlflow.sklearn
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('ruralwatch-models')

print('Imports OK')
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')

2026/04/08 11:45:29 INFO mlflow.tracking.fluent: Experiment with name 'ruralwatch-models' does not exist. Creating a new experiment.


Imports OK
MLflow tracking URI: file:../mlruns


In [6]:
# Cell 6: Train three models with MLflow tracking

# Val and test sets are already clean — no scaling needed for tree models.
# Logistic Regression requires scaling; we fit the scaler on SMOTE'd train only.

results = {}  # Store val metrics for model selection after this cell

# ── MODEL 1: Logistic Regression (interpretable baseline) ─────────────────
with mlflow.start_run(run_name='logistic_regression_v1'):
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('smote', True)
    mlflow.log_param('features', FEATURE_COLS)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_sm)
    X_val_sc   = scaler.transform(X_val)
    X_test_sc  = scaler.transform(X_test)

    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(X_train_sc, y_train_sm)

    y_proba_val_lr = lr.predict_proba(X_val_sc)[:, 1]
    auc_lr   = roc_auc_score(y_val, y_proba_val_lr)
    prauc_lr = average_precision_score(y_val, y_proba_val_lr)

    mlflow.log_metric('val_roc_auc', auc_lr)
    mlflow.log_metric('val_pr_auc', prauc_lr)
    mlflow.sklearn.log_model(lr, 'model')
    mlflow.sklearn.log_model(scaler, 'scaler')

    results['LogisticRegression'] = {'val_roc_auc': auc_lr, 'val_pr_auc': prauc_lr}
    print(f'LR  | Val ROC-AUC: {auc_lr:.4f} | Val PR-AUC: {prauc_lr:.4f}')

# ── MODEL 2: Random Forest ─────────────────────────────────────────────────
with mlflow.start_run(run_name='random_forest_v1'):
    mlflow.log_param('model_type', 'RandomForest')
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('max_depth', 12)
    mlflow.log_param('smote', True)

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=12,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf.fit(X_train_sm, y_train_sm)

    y_proba_val_rf = rf.predict_proba(X_val)[:, 1]
    auc_rf   = roc_auc_score(y_val, y_proba_val_rf)
    prauc_rf = average_precision_score(y_val, y_proba_val_rf)

    mlflow.log_metric('val_roc_auc', auc_rf)
    mlflow.log_metric('val_pr_auc', prauc_rf)
    mlflow.sklearn.log_model(rf, 'model')

    results['RandomForest'] = {'val_roc_auc': auc_rf, 'val_pr_auc': prauc_rf}
    print(f'RF  | Val ROC-AUC: {auc_rf:.4f} | Val PR-AUC: {prauc_rf:.4f}')

# ── MODEL 3: XGBoost ──────────────────────────────────────────────────────
with mlflow.start_run(run_name='xgboost_v1'):
    mlflow.log_param('model_type', 'XGBoost')
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('learning_rate', 0.05)
    mlflow.log_param('max_depth', 5)
    mlflow.log_param('smote', True)

    # scale_pos_weight: upweights the positive class during training.
    # Computed from the SMOTE'd set — after SMOTE it's 1.0, but we keep
    # this pattern explicit so the parameter is logged correctly.
    pos_weight = (y_train_sm == 0).sum() / (y_train_sm == 1).sum()

    xgb = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=pos_weight,
        eval_metric='aucpr',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    xgb.fit(
        X_train_sm, y_train_sm,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_proba_val_xgb = xgb.predict_proba(X_val)[:, 1]
    auc_xgb   = roc_auc_score(y_val, y_proba_val_xgb)
    prauc_xgb = average_precision_score(y_val, y_proba_val_xgb)

    mlflow.log_metric('val_roc_auc', auc_xgb)
    mlflow.log_metric('val_pr_auc', prauc_xgb)
    mlflow.sklearn.log_model(xgb, 'model')

    results['XGBoost'] = {'val_roc_auc': auc_xgb, 'val_pr_auc': prauc_xgb}
    print(f'XGB | Val ROC-AUC: {auc_xgb:.4f} | Val PR-AUC: {prauc_xgb:.4f}')

# ── SUMMARY ───────────────────────────────────────────────────────────────
print('\n=== MODEL COMPARISON (Validation Set) ===')
for model_name, metrics in results.items():
    print(f'{model_name:<20} ROC-AUC: {metrics["val_roc_auc"]:.4f} | PR-AUC: {metrics["val_pr_auc"]:.4f}')

2026/04/08 11:46:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 11:46:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/04/08 11:46:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 11:46:20 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/04/08 11:46:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


LR  | Val ROC-AUC: 0.6396 | Val PR-AUC: 0.0018


2026/04/08 11:46:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 11:46:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RF  | Val ROC-AUC: 0.7124 | Val PR-AUC: 0.0164


2026/04/08 11:46:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 11:46:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGB | Val ROC-AUC: 0.7375 | Val PR-AUC: 0.3372

=== MODEL COMPARISON (Validation Set) ===
LogisticRegression   ROC-AUC: 0.6396 | PR-AUC: 0.0018
RandomForest         ROC-AUC: 0.7124 | PR-AUC: 0.0164
XGBoost              ROC-AUC: 0.7375 | PR-AUC: 0.3372


In [7]:
# Cell 7: Final test set evaluation + SHAP explainability

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — required for plt.savefig in Jupyter
import matplotlib.pyplot as plt

os.makedirs('../docs', exist_ok=True)

best_model = xgb

# ── FINAL TEST SET EVALUATION ─────────────────────────────────────────────
y_proba_test = best_model.predict_proba(X_test)[:, 1]

test_roc_auc = roc_auc_score(y_test, y_proba_test)
test_pr_auc  = average_precision_score(y_test, y_proba_test)

print('=== FINAL TEST SET EVALUATION (XGBoost) ===')
print(f'ROC-AUC : {test_roc_auc:.4f}')
print(f'PR-AUC  : {test_pr_auc:.4f}')

# Log test metrics to the xgboost_v1 MLflow run
# We need the run ID — retrieve the most recent completed run
runs = mlflow.search_runs(experiment_names=['ruralwatch-models'])
xgb_run_id = runs[runs['tags.mlflow.runName'] == 'xgboost_v1'].iloc[0]['run_id']

with mlflow.start_run(run_id=xgb_run_id):
    mlflow.log_metric('test_roc_auc', test_roc_auc)
    mlflow.log_metric('test_pr_auc', test_pr_auc)

print(f'\nTest metrics logged to MLflow run: {xgb_run_id[:8]}...')

# ── CLASSIFICATION REPORT AT THRESHOLD 0.4 ────────────────────────────────
# Default threshold of 0.5 will miss most positives at this imbalance ratio.
# 0.4 trades some precision for better recall — appropriate for an early
# warning system where missing a closure (false negative) is more costly
# than a false alarm (false positive).
y_pred_test = (y_proba_test >= 0.4).astype(int)
print('\n=== CLASSIFICATION REPORT (threshold = 0.4) ===')
print(classification_report(y_test, y_pred_test, target_names=['Stable', 'Distress']))

# ── SHAP GLOBAL FEATURE IMPORTANCE ────────────────────────────────────────
print('\nComputing SHAP values...')
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# Convert X_test to DataFrame with column names for readable plot labels
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLS)

fig, ax = plt.subplots(figsize=(9, 5))
shap.summary_plot(
    shap_values, X_test_df,
    feature_names=FEATURE_COLS,
    show=False
)
plt.title('SHAP Feature Importance — RuralWatch (XGBoost)', fontsize=12)
plt.tight_layout()
plt.savefig('../docs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print('SHAP summary plot saved to docs/shap_summary.png')

=== FINAL TEST SET EVALUATION (XGBoost) ===
ROC-AUC : 0.8789
PR-AUC  : 0.0726

Test metrics logged to MLflow run: 9d312952...

=== CLASSIFICATION REPORT (threshold = 0.4) ===
              precision    recall  f1-score   support

      Stable       1.00      0.98      0.99      3626
    Distress       0.05      0.33      0.09        12

    accuracy                           0.98      3638
   macro avg       0.53      0.66      0.54      3638
weighted avg       0.99      0.98      0.99      3638


Computing SHAP values...
SHAP summary plot saved to docs/shap_summary.png


In [9]:
# Cell 8: Fairness analysis — false positive rate by rural level

# We compare FPR across the three rural tiers present in the test set.
# An early warning system that over-flags Remote Rural hospitals wastes
# scarce state intervention resources. Unequal FPR across tiers is an
# actionable fairness concern analogous to the racial FPR disparity in RecidivAI.

y_test_aligned    = y_test.reset_index(drop=True)
y_pred_aligned    = pd.Series(y_pred_test).reset_index(drop=True)
y_proba_aligned   = pd.Series(y_proba_test).reset_index(drop=True)
rural_aligned     = rural_test.reset_index(drop=True)

print('=== FAIRNESS ANALYSIS: METRICS BY RURAL CLASSIFICATION LEVEL ===\n')
print(f'{"Rural Level":<20} {"N":>6} {"Positives":>10} {"FPR":>8} {"Recall":>8} {"PR-AUC":>8}')
print('-' * 65)

for level in ['Micropolitan', 'Small Town', 'Remote Rural']:
    mask = rural_aligned == level
    if mask.sum() == 0:
        continue

    yt = y_test_aligned[mask]
    yp = y_pred_aligned[mask]
    yproba = y_proba_aligned[mask]

    n         = mask.sum()
    positives = yt.sum()

    # False Positive Rate: of all truly stable hospitals, what fraction
    # did we incorrectly flag as distressed?
    fp  = ((yt == 0) & (yp == 1)).sum()
    tn  = ((yt == 0) & (yp == 0)).sum()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')

    # Recall: of all hospitals that truly closed, what fraction did we catch?
    tp  = ((yt == 1) & (yp == 1)).sum()
    fn  = ((yt == 1) & (yp == 0)).sum()
    recall = tp / (tp + fn) if (tp + fn) > 0 else float('nan')

    # PR-AUC by tier — only meaningful where positives > 1
    if positives > 1:
        tier_prauc = average_precision_score(yt, yproba)
    else:
        tier_prauc = float('nan')  # Not computable with 0 or 1 positive

    print(f'{level:<20} {n:>6} {positives:>10} {fpr:>8.3f} {recall:>8.3f} {tier_prauc:>8.4f}')

print('\nNote: PR-AUC shown as nan where tier has 0 or 1 positive-class rows.')
print('FPR disparity across tiers indicates differential alert burden by rurality.')

=== FAIRNESS ANALYSIS: METRICS BY RURAL CLASSIFICATION LEVEL ===

Rural Level               N  Positives      FPR   Recall   PR-AUC
-----------------------------------------------------------------
Micropolitan            905          4    0.014    0.500   0.2676
Small Town             2066          8    0.026    0.250   0.0326
Remote Rural            667          0    0.009      nan      nan

Note: PR-AUC shown as nan where tier has 0 or 1 positive-class rows.
FPR disparity across tiers indicates differential alert burden by rurality.


In [10]:
# Cell 9: SHAP waterfall plot for a single hospital

# Find the highest-risk hospital in the test set — the one the model is
# most confident about. This is the example we'll show in the README
# and explain in the dashboard.

highest_risk_idx = y_proba_aligned.idxmax()

print(f'Highest-risk hospital in test set:')
print(f'  Index         : {highest_risk_idx}')
print(f'  Risk score    : {y_proba_aligned[highest_risk_idx]:.4f}')
print(f'  True label    : {y_test_aligned[highest_risk_idx]} '
      f'({"Closed" if y_test_aligned[highest_risk_idx] == 1 else "Stable"})')
print(f'  Rural level   : {rural_aligned[highest_risk_idx]}')
print()

# Pull the SHAP values for this single hospital
# shap_values is shape (n_test_rows, n_features)
sv_single = shap_values[highest_risk_idx]

# Build a shap.Explanation object — required by waterfall_plot
explanation = shap.Explanation(
    values       = sv_single,
    base_values  = explainer.expected_value,
    data         = X_test_df.iloc[highest_risk_idx].values,
    feature_names= FEATURE_COLS
)

fig, ax = plt.subplots(figsize=(9, 5))
shap.waterfall_plot(explanation, show=False)
plt.title(
    f'SHAP Waterfall — Highest-Risk Hospital '
    f'(Score: {y_proba_aligned[highest_risk_idx]:.3f})',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../docs/shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.close()
print('SHAP waterfall saved to docs/shap_waterfall_example.png')

Highest-risk hospital in test set:
  Index         : 2569
  Risk score    : 0.9930
  True label    : 0 (Stable)
  Rural level   : Micropolitan

SHAP waterfall saved to docs/shap_waterfall_example.png
